# SCORE-Net training — German

This notebook trains one model and writes all artifacts to its own folder:

* `global-best-weights*.pth` — model with the best **validation** accuracy across pruning.
* `final-iter-best-weights*.pth` — final pruning iteration (highest sparsity).
* `training_history*.json` — sparsity, validation accuracy, and rule count for each iteration.
* `*-pruned.pth` — compacted copies after compression.

The test set is evaluated only once, at the end; early stopping and model selection use `val_loader`.

## Dataset-specific protocol

German Credit has only 1,000 examples. A 72/8/20 split leaves a validation set
of 80 examples (one example = 1.25 points), which is too coarse for early
stopping and checkpoint selection. German therefore uses an **80/10/10** split,
and model selection uses a **smoothed** rule (3-iteration moving average plus a
compactness tie-breaker). Split ratios and the selection rule are read from the
`DatasetSpec` entry in `utils/experiments.py`; no second constant is kept here.


In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# Locate the project root regardless of the notebook's working directory.
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "utils" / "experiments.py").exists()),
    cwd.parent,
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.reproducibility import set_seed
from utils.experiments import DATASETS, load_splits
from utils.model import SCORENet
from utils.train import run_iterative_pruning_pipeline
from utils.custom_utils import prune_and_compact_model

# Seed all randomness sources (Python, NumPy, PyTorch, CUDA).
SEED = set_seed(42)

if torch.cuda.is_available():
    device = "cuda"
    print(f"CUDA available! GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
elif torch.backends.mps.is_available():
    device = "mps"
    print("MPS available! GPU")
else:
    device = "cpu"
    print("CUDA not found. Running on CPU.")


CUDA available! GPU: NVIDIA GeForce GTX 1650
GPU count: 1


In [2]:
# ==============================================================================
# EXPERIMENT CONFIGURATION (single source: utils/experiments.py → DatasetSpec)
# ==============================================================================
SPEC = DATASETS["german"]

BATCH_SIZE = SPEC.batch_size
H_HIDDEN = SPEC.h_hidden
DROPOUT = SPEC.dropout
EXPECTED_N_FEATURES = 48

# German uses "smoothed" selection: on the small validation set, use a
# 3-iteration moving average instead of one lucky spike; ties go to the
# model with fewer active rules.
SELECTION = SPEC.selection
SMOOTHING_WINDOW = SPEC.smoothing_window

# Artifacts always go to this notebook's folder, independent of cwd.
ARTIFACT_DIR = project_root / "models/german"
HISTORY_PATH = ARTIFACT_DIR / "training_history.json"
GLOBAL_CHECKPOINT = ARTIFACT_DIR / "global-best-weights.pth"
FINAL_CHECKPOINT = ARTIFACT_DIR / "final-iter-best-weights.pth"
GLOBAL_PRUNED = ARTIFACT_DIR / "global-best-weights-pruned.pth"
FINAL_PRUNED = ARTIFACT_DIR / "final-iter-best-weights-pruned.pth"

print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Protocol        : {SPEC.protocol_note} (test_size={SPEC.test_size:.4f}, val_size={SPEC.val_size:.4f})")
print(f"H (initial)     : {H_HIDDEN}")
print(f"Dropout         : {DROPOUT}")
print(f"Selection rule  : {SELECTION} (window={SMOOTHING_WINDOW})")


Artifact directory: /home/ahmet/Projects/ScoreNetSingle/models/german
Protocol        : 80/10/10 (test_size=0.1000, val_size=0.1111)
H (initial)     : 64
Dropout         : 0.25
Selection rule  : smoothed (window=3)


In [3]:
# ==============================================================================
# DATA SPLITS (TRAIN / VALIDATION / TEST)
# ==============================================================================
# load_splits: split first, stratify, and fit StandardScaler on train only.
# This keeps validation and test data out of preprocessing.
split = load_splits("german")

X_train_tensor, y_train_tensor = split.X_train, split.y_train
X_val_tensor, y_val_tensor = split.X_val, split.y_val
X_test_tensor, y_test_tensor = split.X_test, split.y_test

N_FEATURES = split.n_features
N_CLASSES = split.n_classes
feature_names, scaler = split.feature_names, split.scaler

if N_FEATURES != EXPECTED_N_FEATURES:
    raise ValueError(
        f"Expected feature count {EXPECTED_N_FEATURES}, data pipeline returned {N_FEATURES} "
        "returned. If preprocessing changed, update this constant and the N value in the paper."
    )

print(f"Train / validation / test: {X_train_tensor.shape[0]} / {X_val_tensor.shape[0]} / {X_test_tensor.shape[0]}")
print(f"Split protocol            : {SPEC.protocol_note}")
print(f"Feature count (N)         : {N_FEATURES}")
print(f"Class count (C)           : {N_CLASSES}")
print(f"Class distribution (train): {torch.bincount(y_train_tensor).tolist()}")

# Seeded generator for reproducible batch order.
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    split.loader("train").dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator
)
val_loader = split.loader("val", batch_size=BATCH_SIZE)
test_loader = split.loader("test", batch_size=BATCH_SIZE)


[Data] Loading German Credit Risk...
-> German Credit ready: train shape: (800, 48), Test Shape: (100, 48), feature count: 48, Val Shape: (100, 48)
Train / validation / test: 800 / 100 / 100
Split protocol            : 80/10/10
Feature count (N)         : 48
Class count (C)           : 2
Class distribution (train): [240, 560]


In [4]:
# Model
# NOTE: data loading and splitting consume the global RNG. Re-seed here so
# model initialization is independent of the data path and reproducible
# (see README, "Reproducing the results").
set_seed(SEED)
model = SCORENet(N_FEATURES, H_HIDDEN, N_CLASSES, dropout_rate=DROPOUT, device=device).to(device)
print(model)


SCORENet(
  (feature_linear): Linear(in_features=48, out_features=64, bias=False)
  (output_linear): Linear(in_features=64, out_features=2, bias=False)
  (dropout): Dropout(p=0.25, inplace=False)
)


In [5]:
# ==============================================================================
# ITERATIVE PRUNING AND RETRAINING
# ==============================================================================
# Model selection and early stopping use only val_loader.
global_best_model, final_iter_model = run_iterative_pruning_pipeline(
    model, train_loader, val_loader, device,
    prune_amount_feature=SPEC.prune_amount,
    num_prune_iters=SPEC.prune_iters,
    history_path=str(HISTORY_PATH),
    selection=SELECTION,
    smoothing_window=SMOOTHING_WINDOW,
)



=============== INITIAL TRAINING (BASE MODEL) STARTING ===============


Epoch [001/250] | Train Acc: 0.6338 | Val Acc: 0.7100 | LR: 0.001000
Epoch [002/250] | Train Acc: 0.6987 | Val Acc: 0.7000 | LR: 0.001000
Epoch [003/250] | Train Acc: 0.7025 | Val Acc: 0.7000 | LR: 0.001000
Epoch [004/250] | Train Acc: 0.7037 | Val Acc: 0.7000 | LR: 0.001000
Epoch [005/250] | Train Acc: 0.7113 | Val Acc: 0.7000 | LR: 0.001000
Epoch [006/250] | Train Acc: 0.7225 | Val Acc: 0.7000 | LR: 0.001000


Epoch [007/250] | Train Acc: 0.7312 | Val Acc: 0.7000 | LR: 0.001000
Epoch [008/250] | Train Acc: 0.7488 | Val Acc: 0.7000 | LR: 0.001000
Epoch [009/250] | Train Acc: 0.7538 | Val Acc: 0.7000 | LR: 0.001000
Epoch [010/250] | Train Acc: 0.7963 | Val Acc: 0.6900 | LR: 0.001000
Epoch [011/250] | Train Acc: 0.8187 | Val Acc: 0.6900 | LR: 0.001000
Epoch [012/250] | Train Acc: 0.8263 | Val Acc: 0.6800 | LR: 0.000500


Epoch [013/250] | Train Acc: 0.8712 | Val Acc: 0.6800 | LR: 0.000500
Epoch [014/250] | Train Acc: 0.8725 | Val Acc: 0.6800 | LR: 0.000500
Epoch [015/250] | Train Acc: 0.8750 | Val Acc: 0.6800 | LR: 0.000500
Epoch [016/250] | Train Acc: 0.8850 | Val Acc: 0.6800 | LR: 0.000500
Epoch [017/250] | Train Acc: 0.8862 | Val Acc: 0.6900 | LR: 0.000500
Epoch [018/250] | Train Acc: 0.8950 | Val Acc: 0.7000 | LR: 0.000500


Epoch [019/250] | Train Acc: 0.8975 | Val Acc: 0.6900 | LR: 0.000500
Epoch [020/250] | Train Acc: 0.9163 | Val Acc: 0.6900 | LR: 0.000500
Epoch [021/250] | Train Acc: 0.9213 | Val Acc: 0.6800 | LR: 0.000500
[!] Early stopping triggered (20 epochs without improvement). Best in cycle: 0.7100



[PRUNING] Iteration 1/14: Feature_Linear zeros: %20.0 | Output_Linear zeros: %0.0

=============== RETRAINING (Iteration 1/14) STARTING ===============
Epoch [001/250] | Train Acc: 0.7000 | Val Acc: 0.7000 | LR: 0.001000
Epoch [002/250] | Train Acc: 0.7000 | Val Acc: 0.7000 | LR: 0.001000
Epoch [003/250] | Train Acc: 0.7000 | Val Acc: 0.7000 | LR: 0.001000
Epoch [004/250] | Train Acc: 0.7087 | Val Acc: 0.7000 | LR: 0.001000
Epoch [005/250] | Train Acc: 0.7100 | Val Acc: 0.7000 | LR: 0.001000


... (79 outputs left out)

Epoch [016/250] | Train Acc: 0.7688 | Val Acc: 0.7500 | LR: 0.001000
Epoch [017/250] | Train Acc: 0.7900 | Val Acc: 0.7300 | LR: 0.001000
Epoch [018/250] | Train Acc: 0.7863 | Val Acc: 0.7600 | LR: 0.001000
Epoch [019/250] | Train Acc: 0.7788 | Val Acc: 0.7400 | LR: 0.001000
Epoch [020/250] | Train Acc: 0.7937 | Val Acc: 0.7600 | LR: 0.001000
Epoch [021/250] | Train Acc: 0.7800 | Val Acc: 0.7400 | LR: 0.001000


Epoch [022/250] | Train Acc: 0.7863 | Val Acc: 0.7300 | LR: 0.001000
Epoch [023/250] | Train Acc: 0.7925 | Val Acc: 0.7400 | LR: 0.001000
Epoch [024/250] | Train Acc: 0.7712 | Val Acc: 0.7300 | LR: 0.001000
Epoch [025/250] | Train Acc: 0.7900 | Val Acc: 0.7300 | LR: 0.001000
Epoch [026/250] | Train Acc: 0.7900 | Val Acc: 0.7300 | LR: 0.001000
Epoch [027/250] | Train Acc: 0.7900 | Val Acc: 0.7200 | LR: 0.001000


Epoch [028/250] | Train Acc: 0.7975 | Val Acc: 0.7200 | LR: 0.001000
Epoch [029/250] | Train Acc: 0.7750 | Val Acc: 0.7200 | LR: 0.000500
Epoch [030/250] | Train Acc: 0.7900 | Val Acc: 0.7300 | LR: 0.000500
Epoch [031/250] | Train Acc: 0.7837 | Val Acc: 0.7300 | LR: 0.000500
Epoch [032/250] | Train Acc: 0.7963 | Val Acc: 0.7400 | LR: 0.000500
Epoch [033/250] | Train Acc: 0.7850 | Val Acc: 0.7500 | LR: 0.000500


Epoch [034/250] | Train Acc: 0.8050 | Val Acc: 0.7300 | LR: 0.000500
Epoch [035/250] | Train Acc: 0.7887 | Val Acc: 0.7500 | LR: 0.000500
Epoch [036/250] | Train Acc: 0.7863 | Val Acc: 0.7300 | LR: 0.000500
Epoch [037/250] | Train Acc: 0.7950 | Val Acc: 0.7300 | LR: 0.000500
Epoch [038/250] | Train Acc: 0.7975 | Val Acc: 0.7300 | LR: 0.000500
[!] Early stopping triggered (20 epochs without improvement). Best in cycle: 0.7600
--> [SMOOTHED SELECTION] Iteration 2 selected (Val Acc: 0.7900 | Rules: 64 | Smoothed: 0.7900)

========================= TRAINING COMPLETE =========================
1. Model: best overall (Global Best Val Acc: 0.7900)
2. Model: 14. best of the final iteration (Final Pruned Model)

[HISTORY] Training history saved: /home/ahmet/Projects/ScoreNetSingle/models/german/training_history.json


In [6]:
# Raw (pre-compaction) checkpoints loaded by the analysis notebooks.
torch.save(global_best_model.state_dict(), GLOBAL_CHECKPOINT)
torch.save(final_iter_model.state_dict(), FINAL_CHECKPOINT)
print(f"Saved: {GLOBAL_CHECKPOINT}")
print(f"Saved: {FINAL_CHECKPOINT}")


Saved: /home/ahmet/Projects/ScoreNetSingle/models/german/global-best-weights.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/german/final-iter-best-weights.pth


In [7]:
# ==============================================================================
# COMPACTION (STRUCTURAL + EMPIRICAL DEAD-RULE CLEANUP, EXACT BIAS SHIFTING)
# ==============================================================================
global_best_model = prune_and_compact_model(global_best_model, train_loader)
final_iter_model = prune_and_compact_model(final_iter_model, train_loader)

torch.save(global_best_model.state_dict(), GLOBAL_PRUNED)
torch.save(final_iter_model.state_dict(), FINAL_PRUNED)
print(f"Saved: {GLOBAL_PRUNED}")
print(f"Saved: {FINAL_PRUNED}")



MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 64
-> Structurally empty rules (<= 0.0001) : 0
-> Empirically dead rules (never triggered): 0
Total removed rules: 0 (removal rate: %0.0)
Active surviving rules (new H): 64
No dead or empty rules to remove; the model is already compact.



MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 64
-> Structurally empty rules (<= 0.0001) : 12
-> Empirically dead rules (never triggered): 6
Total removed rules: 18 (removal rate: %28.1)
Active surviving rules (new H): 46
[OK] Pruning and compaction complete.
-> Net bias contribution transferred to classes:
[-0.2482  0.1977]
-> New rule count (H): 46

Saved: /home/ahmet/Projects/ScoreNetSingle/models/german/global-best-weights-pruned.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/german/final-iter-best-weights-pruned.pth


In [8]:
# ==============================================================================
# EVALUATION (test only, once)
# ==============================================================================
global_best_model.eval()
final_iter_model.eval()

with torch.no_grad():
    for model_name, trained_model in [
        ("Final Iter (compacted)", final_iter_model),
        ("Global Best (compacted)", global_best_model),
    ]:
        test_preds = trained_model(X_test_tensor.to(device)).argmax(dim=1).cpu()
        test_acc = (test_preds == y_test_tensor).float().mean().item()
        val_preds = trained_model(X_val_tensor.to(device)).argmax(dim=1).cpu()
        val_acc = (val_preds == y_val_tensor).float().mean().item()
        print(
            f"{model_name:26s} | H={trained_model.H:5d} | "
            f"Val: {val_acc:.4f} | Test: {test_acc:.4f}"
        )


Final Iter (compacted)     | H=   46 | Val: 0.7600 | Test: 0.7500
Global Best (compacted)    | H=   64 | Val: 0.7900 | Test: 0.7600
